# KF_Net: training experiments

Runs `main` or one experiment branch (`exp/...`) of https://github.com/elasphin/KF_Net on Colab or Kaggle:
training (`train.py`), test (`test.py`), figures (`show_results.py`) and the comparison of every
experiment run so far in the same mode (`show_results.py compare`).

1. Choose the branch and the mode in the next cell (and the option of `exp/input-norm`, `exp/bptt-kalmannet` or `exp/lr-sweep`).
2. Run all cells (Runtime → Run all).
3. If the session ends during the training, run all cells again with the same choices: the training continues
   after its last saved epoch (`train.py`).

| Mode | Data (first fusion epochs: Data01 / Data02) | Epochs | Use |
|---|---|---|---|
| `quick` | 1500 / 600 | 3 | does the branch run? |
| `screening` | 3000 / 1000 | 100 | comparison of the branches (the same for all) |
| `full` | all / all | 480 | as in the paper |

Outputs: `My Drive/KF_Net_outputs/<mode>/<experiment>` on Colab (the dataset in `My Drive/Dataset`),
`/kaggle/working/outputs/<mode>/<experiment>` on Kaggle (the dataset attached to the notebook, Internet on).

In [ ]:
# @title Experiment
BRANCH = 'main'  # @param ["main", "exp/alternating", "exp/grad-clip", "exp/gain-scale", "exp/lr-sweep", "exp/input-norm", "exp/bptt-kalmannet", "exp/tbptt-sensorfusion"]
MODE = 'quick'  # @param ["quick", "screening", "full"]
INPUT_NORMALIZATION = 'zscore'  # @param ["zscore", "l2"]
GAIN_SCALE = True  # @param {type:"boolean"}
LEARNING_RATE = 0.01  # @param {type:"number"}
RESTART = False  # @param {type:"boolean"}
# INPUT_NORMALIZATION: exp/input-norm only. GAIN_SCALE: exp/bptt-kalmannet only (False diverges).
# LEARNING_RATE: exp/lr-sweep only (one rate per run, each in its own folder; Fig. 15: 0.001 ... 0.02).
# RESTART: a new training even if a saved one exists.

# Data (first fusion epochs of each dataset, None = all) and training epochs of each mode, the same for every branch
MODES = {
    'quick': {'train': 1500, 'test': 600, 'epochs': 3},
    'screening': {'train': 3000, 'test': 1000, 'epochs': 100},
    'full': {'train': None, 'test': None, 'epochs': 480},
}

In [ ]:
# @title Code of the branch and Python packages
import os
import re
import subprocess

KAGGLE = os.path.isdir('/kaggle')
if not KAGGLE:
    from google.colab import drive
    drive.mount('/content/drive')
folder = f"{'/tmp' if KAGGLE else '/content'}/KF_Net_{BRANCH.split('/')[-1]}"
if not os.path.isdir(folder):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/elasphin/KF_Net.git', folder], check=True)
os.chdir(folder)
subprocess.run(['git', 'fetch', '-q', 'origin', BRANCH], check=True)
subprocess.run(['git', 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)   # latest code, settings as committed
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
# @title Settings of the mode
mode = MODES[MODE]
text = open('settings.py').read()


def put(name, value):
    global text
    text, count = re.subn(rf'^{name} = .*$', f'{name} = {value!r}', text, flags=re.M)
    if count != 1:
        raise ValueError(f'{name} not found in settings.py of {BRANCH}')


put('MAX_FUSION_EPOCHS', {'train': mode['train'], 'test': mode['test']})
put('TRAINING_EPOCHS', mode['epochs'])
if BRANCH == 'exp/bptt-kalmannet':
    put('WARMUP_EPOCHS', round(mode['epochs'] * 5 / 6))       # the share of the branch: 400 of 480
    put('GAIN_SCALE', GAIN_SCALE)
if BRANCH == 'exp/input-norm':
    put('INPUT_NORMALIZATION', INPUT_NORMALIZATION)
if BRANCH == 'exp/lr-sweep':
    put('LEARNING_RATE', LEARNING_RATE)
for old in ("Path('/content/drive/MyDrive/KF_Net_outputs')", "Path('/kaggle/working/outputs')"):
    if old not in text:
        raise ValueError(f'{old} not found in settings.py of {BRANCH}')
    text = text.replace(old, f"{old[:-2]}/{MODE}')")        # each mode in its own folder
open('settings.py', 'w').write(text)
!python settings.py

In [ ]:
# @title Training, test and figures (an interrupted training continues: run all cells again)
restart = '--restart' if RESTART else ''
!python train.py {restart} && python test.py && python show_results.py

In [ ]:
# @title Comparison of every experiment of this mode
from IPython.display import Image, display

!python show_results.py compare
root = subprocess.run(['python', '-c', 'import settings; print(settings.OUTPUT_ROOT)'], capture_output=True,
                      text=True, check=True).stdout.split()[-1]
if os.path.exists(f'{root}/comparison.png'):
    display(Image(f'{root}/comparison.png'))